In [17]:
import numpy as np
import pandas as pd
from collections import Counter

pos tgging: giving correct grammatical category to the words
NN	  Noun
NNP	  Proper noun
PSP	  Postposition
JJ	  Adjective
VM	  Main verb
VAUX  Auxiliary verb

In [18]:
sentence = "सीता बगीचे से सुंदर लाल फूल तोड़ती है"
tags = ["NNP", "NN", "PSP", "JJ", "VM", "VAUX"]
my_tags = ["NNP", "NN", "PSP", "JJ", "JJ", "NN", "VM", "VAUX"]
tok = sentence.split()
unique_words = set(tok)

print(tok)
print("Observations:", tok)
print("Hidden States:", my_tags)
print("Number of words:", len(tok))
print("Number of tags:", len(my_tags))

['सीता', 'बगीचे', 'से', 'सुंदर', 'लाल', 'फूल', 'तोड़ती', 'है']
Observations: ['सीता', 'बगीचे', 'से', 'सुंदर', 'लाल', 'फूल', 'तोड़ती', 'है']
Hidden States: ['NNP', 'NN', 'PSP', 'JJ', 'JJ', 'NN', 'VM', 'VAUX']
Number of words: 8
Number of tags: 8


In [19]:
trans_prob = np.zeros((6, 6))

for i in range(0, len(my_tags)-1):
    for j in range(0, 6):
        if tags[j] == my_tags[i]:
            break
    for k in range(0, 6):
        if tags[k] == my_tags[i+1]:
            break

    trans_prob[j][k] += 1
print(trans_prob)

[[0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 1. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0.]]


In [20]:
prev = 0
trans_prob_list = []
for tg in my_tags:
    for k in range(0, 6):
        if tags[k] == tg:
            break
    if prev != 0:
        for j in range(0, 6):
            if tags[j] == prev:
                break
        tot = sum(trans_prob[j])
        print(prev, " ", tg, " ", trans_prob[j][k]/ tot)
        trans_prob_list.append([prev, tg, trans_prob[j][k]/ tot])
    else:
        print(prev, " ", tg, " ", 1)
        trans_prob_list.append([prev, tg, 1])
    prev = tg
print(tg, " ", 1, " ", 1)
trans_prob_list.append([tg, 1, 1])

print(trans_prob_list)

0   NNP   1
NNP   NN   1.0
NN   PSP   0.5
PSP   JJ   1.0
JJ   JJ   0.5
JJ   NN   0.5
NN   VM   0.5
VM   VAUX   1.0
VAUX   1   1
[[0, 'NNP', 1], ['NNP', 'NN', np.float64(1.0)], ['NN', 'PSP', np.float64(0.5)], ['PSP', 'JJ', np.float64(1.0)], ['JJ', 'JJ', np.float64(0.5)], ['JJ', 'NN', np.float64(0.5)], ['NN', 'VM', np.float64(0.5)], ['VM', 'VAUX', np.float64(1.0)], ['VAUX', 1, 1]]


In [23]:
emis_prob = np.array((6, len(unique_words)))
emis_prob_list = []

word_count = Counter(tok)
counter_tag = Counter(my_tags)

for word in unique_words:
    for j in range(0, len(tok)):
            if word == tok[j]:
                break
    emis_prob_list.append([word, my_tags[j], word_count[word]/ counter_tag[my_tags[j]]])
print(emis_prob_list)

[['फूल', 'NN', 0.5], ['लाल', 'JJ', 0.5], ['सुंदर', 'JJ', 0.5], ['बगीचे', 'NN', 0.5], ['से', 'PSP', 1.0], ['सीता', 'NNP', 1.0], ['तोड़ती', 'VM', 1.0], ['है', 'VAUX', 1.0]]


In [26]:
prob = 1
for p in emis_prob_list:
    prob = prob * p[2]
print(prob)
prob2 = 1
for p in trans_prob_list:
    prob2 = prob2 * p[2]
print(prob2)

print(prob*prob2)

0.0625
0.0625
0.00390625
